# Carga de provisiones

Proceso de consolidacion de provisiones por cliente hacia la capa gold.

## 1. Cabecera
> **Descripcion:** Informacion general del proceso: objetivo, version, responsable y tablas involucradas.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Riesgos - Provisiones
# PROCESO        : ETL_PROVISIONES
# OBJETIVO       : Consolidar las provisiones vigentes por cliente
# VERSION        : 1.0.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 28/08/2026
# TABLA FUENTE   : mb_silver_prod.rie.h_provision
# TABLA DESTINO  : mb_gold_prod.riesgos.fct_provision
# FRECUENCIA     : Diaria
# -------------------------------------------------------------------------

## 2. Importacion de librerias
> **Descripcion:** Librerias estandar, de terceros y locales, en ese orden.

In [ ]:
import logging
import time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Lectura de parametros
> **Descripcion:** Parametros de ejecucion recibidos por widgets, para que el mismo codigo corra en cualquier ambiente.

In [ ]:
dbutils.widgets.text("p_fecha_proceso", "")
dbutils.widgets.text("p_catalogo", "")

var_fecha_proceso = dbutils.widgets.get("p_fecha_proceso")
var_catalogo = dbutils.widgets.get("p_catalogo")

logger = logging.getLogger("ETL_PROVISIONES")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_PROVISIONES")
logger.info("Parametros recibidos por widgets: fecha=%s catalogo=%s",
            var_fecha_proceso, var_catalogo)

## 4. Seccion constantes
> **Descripcion:** Valores que se mantienen constantes a lo largo del proceso.

In [ ]:
TBL_PROVISION_ORIGEN = f"{var_catalogo}.rie.h_provision"
TBL_PROVISION_FINAL = f"{var_catalogo}.riesgos.fct_provision"

EST_VIGENTE = "VIGENTE"
FORMATO_FECHA = "yyyy-MM-dd"

## 5. Funciones de transformacion
> **Descripcion:** Funciones modularizadas de lectura, transformacion y escritura.

In [ ]:
def read_provision_vigente(tabla, fecha):
    """Lee las provisiones vigentes proyectando solo las columnas necesarias."""
    return (
        spark.table(tabla)
        .select("cod_cliente", "mto_provision", "est_provision", "fec_proceso")
        .filter(F.col("fec_proceso") == fecha)
        .filter(F.col("est_provision") == EST_VIGENTE)
    )


def add_tramo_provision(df_origen):
    """Clasifica la provision en tramos segun el monto."""
    return df_origen.withColumn(
        "des_tramo",
        F.when(F.col("mto_provision") >= 10000, F.lit("ALTO"))
        .when(F.col("mto_provision") >= 1000, F.lit("MEDIO"))
        .otherwise(F.lit("BAJO"))
    )

## 6. Logica del proceso
> **Descripcion:** Orquestacion de las funciones definidas previamente.

In [ ]:
ini_etapa = time.perf_counter()

df_provision = read_provision_vigente(TBL_PROVISION_ORIGEN, var_fecha_proceso)
df_provision_tramo = add_tramo_provision(df_provision)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Deduplicacion
> **Descripcion:** Logica de deduplicacion segun las llaves de la tabla.

In [ ]:
ventana_cliente = Window.partitionBy("cod_cliente").orderBy(
    F.col("fec_proceso").desc()
)

df_provision_unica = (
    df_provision_tramo
    .withColumn("nro_orden", F.row_number().over(ventana_cliente))
    .filter(F.col("nro_orden") == 1)
    .drop("nro_orden")
)

## 8. Escritura en la tabla final
> **Descripcion:** Persistencia del resultado en formato Delta.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_provision_unica
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_PROVISION_FINAL)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_PROVISION_FINAL, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 9. Registro de la ejecucion
> **Descripcion:** Cierre del proceso con el registro de duracion y volumen.

In [ ]:
logger.info("Fin del proceso ETL_PROVISIONES. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)